# Korn2018: reproduserbart følgehefte
Sett CEREAL_INPUT_MANIFEST til den private, SHA256-bundne input-manifest.json. Kjør fra repository. Ingen nettverk/DB.
Basal2015–2017 er årsgjennomsnitt; C1110 for norsk hvete beholdes separat fra C1100. Alle tall er beskrivende.

In [1]:
import os, json, hashlib, importlib.util
from pathlib import Path
repo = Path.cwd()
while not (repo / 'scripts/analyze-beredskap-cereals.py').is_file():
    if repo.parent == repo: raise RuntimeError('Run inside Food Systems repository')
    repo = repo.parent
manifest_path = Path(os.environ['CEREAL_INPUT_MANIFEST'])
manifest = json.loads(manifest_path.read_text())
spec = importlib.util.spec_from_file_location('cereals', repo / 'scripts/analyze-beredskap-cereals.py')
module = importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
inputs = []
for key in ('panel', 'norway_common_wheat'):
    entry = manifest[key]; raw = (manifest_path.parent / entry['path']).read_bytes()
    assert hashlib.sha256(raw).hexdigest() == entry['sha256'], 'Source changed'
    inputs.append(json.loads(raw))
analysis = module.analyze(*inputs)
assert len(analysis['records']) == 48
for row in analysis['contrasts']:
    if row['year'] == 2018:
        print(row['country'], row['crop_code'], round(row['production_kt_change_pct'], 3))
print('QA warnings:', len(analysis['warnings']))


NO C1110 -65.727
NO C1300 -23.088
SE C1100 -48.511
SE C1300 -32.241
DK C1100 -43.377
DK C1300 -11.365
FI C1100 -43.271
FI C1300 -12.983
QA warnings: 9
